# AR-SSL4M Pretraining with Mixed Data Sources

This notebook handles the setup and pretraining of the AR-SSL4M model using:
- LIDC data from Google Cloud Storage
- BraTS data from Google Cloud Storage 
- DeepLesion data from Google Drive

In [1]:
# Mount Google Drive (for DeepLesion data only) and authenticate GCS
from google.colab import drive
from google.colab import auth

drive.mount('/content/drive')
auth.authenticate_user()  # 验证 GCS 权限

Mounted at /content/drive


In [2]:
# Configure GCS and check data sources
import os

# 项目 ID 和 GCS 配置
PROJECT_ID = 'brats-preprocess'
GCS_LIDC_BUCKET = "gs://brats_preprocessed_npy"  # LIDC 数据在 GCS
GCS_BRATS_BUCKET = "gs://brats_preprocessed_npy"  # BraTS 数据在 GCS
DEEPLESION_PATH = '/content/drive/MyDrive/dataset/pretrain/DeepLesion/npy'  # DeepLesion 在 Drive

# 设置 GCP 项目
!gcloud config set project {PROJECT_ID}

# 检查 DeepLesion 数据
if os.path.exists(DEEPLESION_PATH):
    deeplesion_files = [f for f in os.listdir(DEEPLESION_PATH) if f.endswith('.npy')]
    print(f"DeepLesion found: {len(deeplesion_files)} files at {DEEPLESION_PATH}")
else:
    print(f"DeepLesion NOT found at {DEEPLESION_PATH}")

# 详细检查 GCS 数据结构
print(f"\n=== 详细检查 GCS 存储桶结构 ===")
print(f"存储桶: {GCS_LIDC_BUCKET}")

# 1. 检查顶层目录结构
print("\n1. 顶层目录结构:")
!gsutil ls {GCS_LIDC_BUCKET}

# 2. 检查是否有直接的 .npy 文件
print("\n2. 检查顶层是否有 .npy 文件:")
!gsutil ls {GCS_LIDC_BUCKET}/*.npy 2>/dev/null | head -5 || echo "顶层没有 .npy 文件"

# 3. 搜索所有 .npy 文件 (前20个)
print("\n3. 搜索所有 .npy 文件 (前20个):")
!gsutil ls -r {GCS_LIDC_BUCKET}/**/*.npy 2>/dev/null | head -20 || echo "没有找到 .npy 文件"

# 4. 统计总文件数
print("\n4. 统计 .npy 文件总数:")
!gsutil ls -r {GCS_LIDC_BUCKET}/**/*.npy 2>/dev/null | wc -l || echo "0"

print(f"\n=== GCS 检查完成 ===")

['LIDC-IDRI', 'AR-SSL4M-DEMO', 'patch_random_spatial', 'Untitled folder', 'colab_train_list.txt', 'output']
Dataset found at /content/drive/MyDrive/dataset/LIDC-IDRI
['LIDC-IDRI', 'AR-SSL4M-DEMO', 'patch_random_spatial', 'Untitled folder', 'colab_train_list.txt', 'output']


In [ ]:
# 更详细的 GCS 数据类型检查
print("=== 检查 GCS 中的数据类型 ===")

# 检查是否有 LIDC 相关的文件
print("\n1. 搜索 LIDC 相关文件:")
!gsutil ls -r {GCS_LIDC_BUCKET}/**/*LIDC*.npy 2>/dev/null | head -10 || echo "没有找到 LIDC 相关文件"

# 检查是否有 BraTS 相关的文件
print("\n2. 搜索 BraTS 相关文件:")
!gsutil ls -r {GCS_BRATS_BUCKET}/**/*BraTS*.npy 2>/dev/null | head -10 || echo "没有找到 BraTS 相关文件"
!gsutil ls -r {GCS_BRATS_BUCKET}/**/*GLI*.npy 2>/dev/null | head -10 || echo "没有找到 GLI 相关文件"

# 检查 BraTS 的不同模态
print("\n3. 检查 BraTS 模态文件:")
print("  检查 t1n 文件:")
!gsutil ls -r {GCS_BRATS_BUCKET}/**/*.t1n.npy 2>/dev/null | wc -l
print("  检查 t1c 文件:")
!gsutil ls -r {GCS_BRATS_BUCKET}/**/*.t1c.npy 2>/dev/null | wc -l
print("  检查 t2w 文件:")
!gsutil ls -r {GCS_BRATS_BUCKET}/**/*.t2w.npy 2>/dev/null | wc -l
print("  检查 t2f 文件:")
!gsutil ls -r {GCS_BRATS_BUCKET}/**/*.t2f.npy 2>/dev/null | wc -l

# 检查第一个文件夹的内容
print("\n4. 检查第一个文件夹的内容:")
!gsutil ls {GCS_LIDC_BUCKET} | head -1 | xargs gsutil ls

print("\n=== 数据类型检查完成 ===")

In [4]:
# Generate Training Lists for Mixed Data Sources
# Create separate lists for LIDC (GCS), BraTS (GCS), and DeepLesion (Drive)

import os
import numpy as np
from tqdm import tqdm

# 本地临时目录用于存储训练列表
LOCAL_TEMP_ROOT = "/content/temp_lists"
os.makedirs(LOCAL_TEMP_ROOT, exist_ok=True)

# === 1. 生成 LIDC Spatial 列表 (从 GCS) ===
spatial_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_spatial.txt')
print("Generating LIDC spatial list from GCS...")
# 获取 GCS 中的 LIDC .npy 文件
# 使用递归搜索来找到文件夹中的 .npy 文件
!gsutil ls -r {GCS_LIDC_BUCKET}/**/*.npy > {spatial_list_path} 2>/dev/null || gsutil ls {GCS_LIDC_BUCKET}/*.npy > {spatial_list_path}
with open(spatial_list_path, 'r') as f:
    spatial_files = [line.strip() for line in f if line.strip().endswith('.npy')]
print(f"Found {len(spatial_files)} LIDC spatial files in GCS")

# === 2. 生成 BraTS Contrast 列表 (从 GCS) ===
contrast_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_contrast.txt')
print("\nGenerating BraTS contrast list from GCS...")

# 获取 GCS 中所有的 .npy 文件
print("Scanning GCS bucket for BraTS files...")
# 使用递归搜索来找到文件夹中的 .npy 文件
!gsutil ls -r {GCS_BRATS_BUCKET}/**/*.npy > /tmp/all_brats_files.txt 2>/dev/null || echo "No .npy files found" > /tmp/all_brats_files.txt

# 读取所有文件列表
try:
    with open('/tmp/all_brats_files.txt', 'r') as f:
        all_files = [line.strip() for line in f if line.strip().endswith('.npy')]
    print(f"Found {len(all_files)} total .npy files in GCS")
except FileNotFoundError:
    print("No files found in GCS bucket. Creating empty contrast list.")
    all_files = []

valid_brats_cases = []
if all_files:
    # 根据你之前的预处理代码，BraTS 文件命名格式应该是：
    # {ds_name}_{nii_id}_{patch_idx}.{modality}.npy
    # 例如：BraTS_BraTS-GLI-01234-000_0.t1n.npy
    
    # 提取所有 t1n 文件作为基准
    t1n_files = [f for f in all_files if '.t1n.npy' in f]
    print(f"Found {len(t1n_files)} t1n files")
    
    if t1n_files:
        print(f"Sample t1n files: {t1n_files[:3]}")
        
        # 验证每个 t1n 文件是否有对应的 t1c, t2w, t2f 文件
        all_files_set = set(all_files)
        
        for t1n_file in t1n_files:
            # 获取 base name (去掉 .t1n.npy)
            base = t1n_file.replace('.t1n.npy', '')
            
            # 检查是否存在所有 4 个模态
            required_files = [
                f"{base}.t1n.npy",
                f"{base}.t1c.npy", 
                f"{base}.t2w.npy",
                f"{base}.t2f.npy"
            ]
            
            # 检查所有必需文件是否都存在
            if all(req_file in all_files_set for req_file in required_files):
                valid_brats_cases.append(base)  # 存储 base name
    
    print(f"Found {len(valid_brats_cases)} valid BraTS contrast cases (with all 4 modalities)")
else:
    print("No BraTS files found in GCS")

# 写入 contrast 列表
with open(contrast_list_path, 'w') as f:
    f.write('\n'.join(valid_brats_cases))

if len(valid_brats_cases) == 0:
    print("WARNING: No valid BraTS cases found! Please check:")
    print(f"- GCS bucket: {GCS_BRATS_BUCKET}")
    print(f"- File naming convention matches your preprocessing")
    print("- Bucket contains preprocessed BraTS .npy files")
else:
    print(f"BraTS contrast list saved to: {contrast_list_path}")

# === 3. 生成 DeepLesion Semantic 列表 (从 Drive) ===
semantic_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_semantic.txt')
if os.path.exists(DEEPLESION_PATH):
    print("\nGenerating DeepLesion semantic list from Drive...")
    deeplesion_files = []
    npy_files = [f for f in os.listdir(DEEPLESION_PATH) if f.endswith('.npy')]
    
    print(f"Checking {len(npy_files)} DeepLesion files...")
    for f in tqdm(npy_files):
        full_path = os.path.join(DEEPLESION_PATH, f)
        try:
            # 快速检查文件是否有效
            data = np.load(full_path, mmap_mode='r')
            if data.size > 0:
                deeplesion_files.append(full_path)
        except Exception as e:
            print(f"Skipping corrupted file: {f}")
    
    with open(semantic_list_path, 'w') as f:
        f.write('\n'.join(deeplesion_files))
    print(f"Found {len(deeplesion_files)} valid DeepLesion files")
else:
    print(f"\nDeepLesion path not found: {DEEPLESION_PATH}")
    # 创建空的语义列表
    with open(semantic_list_path, 'w') as f:
        f.write('')
    print("Created empty semantic list")

print(f"\n=== Training Lists Generated ===")
print(f"Spatial (LIDC): {spatial_list_path}")
print(f"Contrast (BraTS): {contrast_list_path}")
print(f"Semantic (DeepLesion): {semantic_list_path}")

Checking 24850 files in /content/drive/MyDrive/dataset/LIDC-IDRI/patch_random_spatial...


100%|██████████| 24850/24850 [4:35:15<00:00,  1.50it/s]


Verification complete.
Valid files: 24850
Corrupted/Empty files removed: 0
Updated training list at: /content/drive/MyDrive/dataset/LIDC-IDRI/colab_train_list.txt


In [3]:
# Clone the repository (if not already present)
# Cloning from your GitHub repository as requested
!git clone https://github.com/tanglehunter00/AR-SSL4M-DEMO.git

# IMPORTANT: If you are running this notebook and the code is NOT on Drive,
# you need to upload the code files to Colab runtime.

project_root = '/content/AR-SSL4M-DEMO'
import os
if os.path.exists(project_root):
    %cd {project_root}
else:
    print("Project root not found. Please clone or upload your code.")

Cloning into 'AR-SSL4M-DEMO'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 290 (delta 14), reused 9 (delta 9), pack-reused 274 (from 1)
Receiving objects: 100% (290/290), 1.39 MiB | 27.36 MiB/s, done.
Resolving deltas: 100% (134/134), done.
/content/AR-SSL4M-DEMO


In [4]:
# Install dependencies
!pip install timm monai transformers fire

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.7 MB/s eta 0:00:00


In [5]:
# Update dataset configuration to use mixed data sources
# Configure paths for LIDC (GCS), BraTS (GCS), and DeepLesion (Drive)

import os

# 配置路径
spatial_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_spatial.txt')
contrast_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_contrast.txt')
semantic_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_semantic.txt')

# 检查生成的列表文件
for name, path in [('Spatial', spatial_list_path), ('Contrast', contrast_list_path), ('Semantic', semantic_list_path)]:
    if os.path.exists(path):
        with open(path, 'r') as f:
            lines = f.readlines()
        print(f"{name} list: {len(lines)} entries")
    else:
        print(f"{name} list: NOT FOUND")

print(f"\nDataset configuration ready for mixed sources:")
print(f"- LIDC Spatial: GCS ({GCS_LIDC_BUCKET})")
print(f"- BraTS Contrast: GCS ({GCS_BRATS_BUCKET})")
print(f"- DeepLesion Semantic: Drive ({DEEPLESION_PATH})")

Created training list at /content/drive/MyDrive/dataset/LIDC-IDRI/colab_train_list.txt with 24850 files.


In [6]:
# Update dataset configuration for mixed data sources
# Configure to use LIDC (GCS), BraTS (GCS), and DeepLesion (Drive)

config_path = 'pretrain/configs/datasets.py'

# 检查是否有 DeepLesion 数据来决定是否启用语义学习
has_semantic = os.path.exists(semantic_list_path) and os.path.getsize(semantic_list_path) > 0
effective_semantic = semantic_list_path if has_semantic else spatial_list_path

new_config_content = f"""
from dataclasses import dataclass

@dataclass
class custom_dataset:
    dataset: str = "custom_dataset"
    file: str = "image_dataset.py"
    train_split: str = "train"
    test_split: str = "validation"
    # Mixed data sources configuration
    spatial_path: str = "{spatial_list_path}"      # LIDC from GCS
    contrast_path: str = "{contrast_list_path}"    # BraTS from GCS  
    semantic_path: str = "{effective_semantic}"    # DeepLesion from Drive
    img_size = [128, 128, 128]
    patch_size = [16, 16, 16]
    attention_type = 'prefix'
    add_series_data = True      # Enable BraTS multi-modal
    add_spatial_data = True     # Enable LIDC spatial
    is_subset = False
    series_length = 4           # BraTS has 4 modalities
"""

with open(config_path, 'w') as f:
    f.write(new_config_content)

print("Updated datasets.py configuration for mixed data sources.")
print(f"- Spatial (LIDC): {spatial_list_path}")
print(f"- Contrast (BraTS): {contrast_list_path}")
print(f"- Semantic (DeepLesion): {effective_semantic}")
print(f"- Series data enabled: True (for BraTS multi-modal)")
print(f"- Semantic data available: {has_semantic}")

Updated datasets.py configuration.


In [9]:
# Configure GCS access for training
# Set up environment variables and authentication for GCS data loading

import os

# 设置环境变量，让训练代码知道使用 GCS
os.environ['USE_GCS'] = 'true'
os.environ['GCS_BUCKET'] = GCS_LIDC_BUCKET.replace('gs://', '')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID

# 安装 GCS 相关依赖
!pip install -q google-cloud-storage

print("GCS configuration completed:")
print(f"- Project ID: {PROJECT_ID}")
print(f"- GCS Bucket: {GCS_LIDC_BUCKET}")
print(f"- Authentication: Completed")
print(f"\nEnvironment variables set:")
print(f"- USE_GCS: {os.environ.get('USE_GCS')}")
print(f"- GCS_BUCKET: {os.environ.get('GCS_BUCKET')}")
print(f"- GOOGLE_CLOUD_PROJECT: {os.environ.get('GOOGLE_CLOUD_PROJECT')}")

# Switch to pretrain directory
%cd pretrain

# Create output directory
!mkdir -p /content/mixed_data_output

# Run training with mixed data sources
# Keep original training logic from colab_pretrain_GCS.ipynb
!python main.py \
    --enable_fsdp False \
    --output_dir /content/mixed_data_output \
    --batch_size_training 128 \
    --num_epochs 5 \
    --save_metrics True \
    --num_workers_dataloader 4

print("\n=== Training completed with mixed data sources ===")
print("Data sources used:")
print(f"- LIDC Spatial: GCS ({GCS_LIDC_BUCKET})")
print(f"- BraTS Contrast: GCS ({GCS_BRATS_BUCKET})")
print(f"- DeepLesion Semantic: Drive ({DEEPLESION_PATH})")
print(f"- Output directory: /content/mixed_data_output")

cp: cannot stat 'pretrain/newModel.py': No such file or directory
[Errno 2] No such file or directory: 'pretrain'
/content/AR-SSL4M-DEMO/pretrain
2026-01-26 21:45:56.481968: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 21:45:56.500074: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769463956.522029    4891 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769463956.528501    4891 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been register